# Google drive mount

In [14]:
# 계정 드라이브와 마운트하기
# 모두 선택해서 동의해야 마운트가 오류없이 됩니다.
# from google.colab import drive
# drive.mount('/content/drive')

# 파일 가져오기

In [15]:
!ls /content/drive/MyDrive/Kaggle01

'ls'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


In [16]:
!unzip /content/drive/MyDrive/Kaggle01/rokey-boot-camp-mini-competition.zip -d data

'unzip'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


In [17]:
!ls data

'ls'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


# 파일 구조 확인

In [18]:
!ls data/train_data

'ls'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


In [19]:
!ls data/test_data

'ls'��(��) ���� �Ǵ� �ܺ� ����, ������ �� �ִ� ���α׷�, �Ǵ�
��ġ ������ �ƴմϴ�.


In [ ]:
# 첫 10개의 학습 데이터 이름과 label 출력
import pandas as pd
train_df = pd.read_csv("data/train_data.csv")
train_df.head(10)

OSError: [Errno 22] Invalid argument: '03_1_project_Rokey_9_AI_Basic_Project\test_data'

In [ ]:
# 각 파일마다 라벨이 무엇인지 나타내는 dictionary 생성
name2label = dict(zip(train_df["name"], train_df["label"]))
print(name2label['03501.png'])

# glob 라이브러리로 훈련데이터에 있는 파일 리스트 출력
import os
data_path = "data"
from glob import glob
glob(f"{data_path}/train_data/*.png")

8


['data/train_data/00574.png',
 'data/train_data/00651.png',
 'data/train_data/04511.png',
 'data/train_data/00829.png',
 'data/train_data/03466.png',
 'data/train_data/02396.png',
 'data/train_data/04685.png',
 'data/train_data/04229.png',
 'data/train_data/01120.png',
 'data/train_data/04148.png',
 'data/train_data/00993.png',
 'data/train_data/01485.png',
 'data/train_data/00024.png',
 'data/train_data/04666.png',
 'data/train_data/04657.png',
 'data/train_data/04786.png',
 'data/train_data/00661.png',
 'data/train_data/04007.png',
 'data/train_data/03473.png',
 'data/train_data/04649.png',
 'data/train_data/02227.png',
 'data/train_data/01890.png',
 'data/train_data/01015.png',
 'data/train_data/03533.png',
 'data/train_data/00439.png',
 'data/train_data/04367.png',
 'data/train_data/02746.png',
 'data/train_data/03253.png',
 'data/train_data/03491.png',
 'data/train_data/01612.png',
 'data/train_data/04090.png',
 'data/train_data/02913.png',
 'data/train_data/00620.png',
 'data/tra

# DataLoader

압축푼 직후에는 파일적용이 되지 않아 FileNotFoundError 오류가 뜰 수 있습니다.

그러한 경우 약간의 대기 시간 이후 다시 실행하면 됩니다.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
import torchvision.transforms as transforms
from torchvision.transforms import v2

from PIL import Image
from tqdm import tqdm
import pandas as pd

ModuleNotFoundError: No module named 'torchvision'

In [ ]:
from glob import glob

# 커스텀 데이터셋 클래스
class MyDataset(Dataset):
    def __init__(self, data_path, transform=None, train=True):
        self.train = train
        train_df = pd.read_csv(f"{data_path}/train_data.csv")

        self.name2label = dict(zip(train_df["name"], train_df["label"]))

        if self.train:
            self.img_path = glob(f"{data_path}/train_data/*.png")
            self.labels =  [self.name2label[d.split("/")[-1]] for d in self.img_path]
        else:
            self.img_path = glob(f"{data_path}/test_data/*.png")

        self.transform = transform

    def __len__(self):
        return len(self.img_path)   

    def __getitem__(self, index):
        img = Image.open(self.img_path[index])
        if img.mode != 'RGB':
            img = img.convert('RGB')

        if self.transform:
            img = self.transform(img)

        if self.train:
            return img, self.labels[index]
        else:
            return img, self.img_path[index].split("/")[-1]

In [ ]:
# 데이터셋 디렉토리 위치 지정
data_path = "./data"

In [ ]:
'''
데이터 전처리
    - transform.Compose에 전처리할 순서를 차례로 지정한 후 리스트 형태로 입력하여 데이터 생성시 설정한 전처리를 적용
    - 여러 가지의 데이터 증강 기법이 들어감
'''
transform =  transforms.Compose([
    # To-do: 증강 기법 적용
    v2.ToTensor(),
    v2.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

test_transform =  transforms.Compose([
    v2.ToTensor(),
    v2.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

train_data = MyDataset(data_path, train=True, transform=transform)
test_data = MyDataset(data_path, train=False, transform=test_transform)

# Split train data into train and validation
train_size = int(len(train_data) * 0.9)
train_data, val_data = torch.utils.data.random_split(train_data, [train_size, len(train_data) - train_size])


train_loader = torch.utils.data.DataLoader(train_data, batch_size=128, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_data, batch_size=128, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_data, batch_size=128, shuffle=False)

In [ ]:
print(train_data[0])
print(len(train_data))

(tensor([[[0.0039, 0.0000, 0.0314,  ..., 0.0000, 0.1059, 0.3098],
         [0.0078, 0.0000, 0.0392,  ..., 0.0000, 0.0118, 0.1961],
         [0.0039, 0.0000, 0.0353,  ..., 0.1059, 0.0196, 0.0118],
         ...,
         [0.2353, 0.1647, 0.1804,  ..., 0.4824, 0.3922, 0.5686],
         [0.3451, 0.1961, 0.1882,  ..., 0.5294, 0.3490, 0.5490],
         [0.3529, 0.2353, 0.1961,  ..., 0.5020, 0.3882, 0.5020]],

        [[0.0000, 0.0000, 0.0275,  ..., 0.0000, 0.1020, 0.2980],
         [0.0000, 0.0000, 0.0275,  ..., 0.0000, 0.0118, 0.1922],
         [0.0000, 0.0000, 0.0314,  ..., 0.1098, 0.0157, 0.0078],
         ...,
         [0.2824, 0.2471, 0.2980,  ..., 0.5176, 0.4353, 0.6118],
         [0.3843, 0.2784, 0.2627,  ..., 0.5686, 0.3922, 0.5882],
         [0.4314, 0.3255, 0.2353,  ..., 0.5451, 0.4275, 0.5373]],

        [[0.0000, 0.0000, 0.0275,  ..., 0.0000, 0.1255, 0.3608],
         [0.0039, 0.0000, 0.0275,  ..., 0.0000, 0.0314, 0.2275],
         [0.0078, 0.0000, 0.0392,  ..., 0.1647, 0.0471, 0

# Model

In [ ]:
from torchvision.models import resnet18

# Torchvision 라이브러리에서 모델 불러오기
model = resnet18(pretrained=False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

In [ ]:
criterion = nn.CrossEntropyLoss() # 바꿔보기
optimizer = torch.optim.SGD(model.parameters(), lr=0.001, momentum=0.9) # 바꿔보기

num_epochs = 5 # 바꿔보기

total_step = len(train_loader)
for epoch in range(num_epochs):
    total_loss = 0

    pbar = tqdm(enumerate(train_loader), total=len(train_loader))

    for i, (images, labels) in pbar:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        pbar.set_description(f'Epoch [{epoch+1}/{num_epochs}], Loss: {round(total_loss / (i+1),4)}')


    model.eval()
    with torch.no_grad():
        correct = 0
        total = 0
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        print(f'Accuracy of the model on the test images: {100 * correct / total} %')


Epoch [1/5], Loss: 2.2763: 100%|██████████| 36/36 [00:09<00:00,  3.63it/s]


Accuracy of the model on the test images: 13.0 %


Epoch [2/5], Loss: 2.171: 100%|██████████| 36/36 [00:06<00:00,  5.90it/s]


Accuracy of the model on the test images: 21.6 %


Epoch [3/5], Loss: 1.9326: 100%|██████████| 36/36 [00:06<00:00,  5.60it/s]


Accuracy of the model on the test images: 28.2 %


Epoch [4/5], Loss: 1.8175: 100%|██████████| 36/36 [00:06<00:00,  5.91it/s]


Accuracy of the model on the test images: 30.2 %


Epoch [5/5], Loss: 1.6632: 100%|██████████| 36/36 [00:06<00:00,  5.56it/s]


Accuracy of the model on the test images: 32.4 %


# Evaluation (Test)

In [ ]:
len(val_loader.dataset)
len(test_loader.dataset)

500

In [ ]:
correct = 0
total = len(val_loader.dataset)

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        correct += torch.sum((predicted == labels)).item()

print(f'Accuracy : {100 * correct / total} %')


Accuracy : 32.4 %


# Make SubmitFile

In [ ]:
import pandas as pd

# 제출 파일 submission.csv 생성
outputs = {
    'Id': [],
    'Prediction': []
}

for images, id in tqdm(test_loader):
    model.eval()
    with torch.no_grad():
        output = model(images.to(device))
        _, predicted = torch.max(output, 1)
        outputs['Prediction'] += predicted.tolist()
        outputs['Id'] += id

df = pd.DataFrame(outputs)

df.to_csv('submission.csv', index=False, columns=['Id', 'Prediction'])

100%|██████████| 63/63 [00:08<00:00,  7.67it/s]


In [ ]:
# 제출파일 다운로드
from google.colab import files

file_path = "submission.csv"
files.download(file_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>